# Эксперимент 2.6: Оптимизационная точность vs качество предсказания

In [ ]:
import urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.datasets import load_svmlight_file
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error

from oracles import LogCoshL2Oracle, ExponentialLossL2Oracle
from optimization import lbfgs

In [ ]:
LIBSVM_BASE = "https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets"
data_dir = Path("data/libsvm")
data_dir.mkdir(parents=True, exist_ok=True)

def ensure_dataset(name, rel_url):
    path = data_dir / name
    if not path.exists():
        req = urllib.request.Request(f"{LIBSVM_BASE}/{rel_url}", headers={"User-Agent": "MetOpt-lab/2.0"})
        with urllib.request.urlopen(req, timeout=300) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

def labels_pm_one(y):
    y = np.asarray(y, dtype=float).ravel()
    u = np.unique(y)
    if len(u) == 2 and 0.0 in u and 1.0 in u:
        return 2.0 * y - 1.0
    return y

reg_path = ensure_dataset("abalone_scale", "regression/abalone_scale")
clf_path = ensure_dataset("a9a", "binary/a9a")

A_reg, y_reg = load_svmlight_file(reg_path)
A_clf, y_clf = load_svmlight_file(clf_path)
A_reg = A_reg.tocsr().astype(np.float64)
A_clf = A_clf.tocsr().astype(np.float64)
y_reg = np.asarray(y_reg, dtype=np.float64).ravel()
y_clf = labels_pm_one(y_clf)

idx_reg = np.arange(A_reg.shape[0])
idx_clf = np.arange(A_clf.shape[0])
tr_reg, te_reg = train_test_split(idx_reg, test_size=0.2, random_state=42)
tr_clf, te_clf = train_test_split(idx_clf, test_size=0.2, random_state=42)

A_reg_tr, A_reg_te = A_reg[tr_reg], A_reg[te_reg]
y_reg_tr, y_reg_te = y_reg[tr_reg], y_reg[te_reg]
A_clf_tr, A_clf_te = A_clf[tr_clf], A_clf[te_clf]
y_clf_tr, y_clf_te = y_clf[tr_clf], y_clf[te_clf]

In [ ]:
def make_oracles_for_train(A_tr_sp, y_train, is_clf):
    def matvec_Ax(x):
        return A_tr_sp.dot(np.asarray(x, dtype=np.float64).ravel())
    def matvec_ATx(v):
        return A_tr_sp.T.dot(np.asarray(v, dtype=np.float64).ravel())
    def matmat_ATsA(s):
        return A_tr_sp.T.dot(sparse.diags(s)).dot(A_tr_sp).toarray()
    regcoef = 1.0 / A_tr_sp.shape[0]
    if is_clf:
        return ExponentialLossL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, y_train, regcoef)
    return LogCoshL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, y_train, regcoef)

oracle_reg = make_oracles_for_train(A_reg_tr, y_reg_tr, is_clf=False)
oracle_clf = make_oracles_for_train(A_clf_tr, y_clf_tr, is_clf=True)
x0_reg = np.zeros(A_reg_tr.shape[1])
x0_clf = np.zeros(A_clf_tr.shape[1])

In [ ]:
ls = {'method': 'Wolfe', 'c1': 1e-4, 'c2': 0.9, 'alpha_0': 1.0}
max_iter = 40
_, _, h_reg = lbfgs(oracle_reg, x0_reg, tolerance=1e-8, max_iter=max_iter, memory_size=10, line_search_options=ls, trace=True)
_, _, h_clf = lbfgs(oracle_clf, x0_clf, tolerance=1e-8, max_iter=max_iter, memory_size=10, line_search_options=ls, trace=True)

In [ ]:
def collect_metric_curve(oracle, x0, is_clf, max_k):
    metric = []
    for k in range(1, max_k + 1):
        xk, _, _ = lbfgs(oracle, x0, tolerance=1e-14, max_iter=k, memory_size=10, line_search_options=ls, trace=False)
        if is_clf:
            pred = A_clf_te.dot(xk)
            y_hat = np.where(pred >= 0, 1, -1)
            metric.append(accuracy_score(y_clf_te, y_hat))
        else:
            pred = A_reg_te.dot(xk)
            metric.append(mean_squared_error(y_reg_te, pred))
    return np.array(metric)

metric_reg = collect_metric_curve(oracle_reg, x0_reg, is_clf=False, max_k=max_iter)
metric_clf = collect_metric_curve(oracle_clf, x0_clf, is_clf=True, max_k=max_iter)

In [ ]:
def plot_dual_axis(hist, metric, metric_name, title):
    g = np.array(hist['grad_norm'])
    it = np.arange(len(g))

    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(it, hist['func'], label='f_train')
    ax1.plot(it, g, label='||grad||')
    ax1.set_yscale('log')
    ax1.set_xlabel('iteration')
    ax1.set_ylabel('train objective / grad norm (log)')
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    if metric.size > 0:
        ax2.plot(np.arange(len(metric)), metric, color='tab:red', label=metric_name)
    ax2.set_ylabel(metric_name)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')
    plt.title(title)
    plt.tight_layout()

plot_dual_axis(h_reg, metric_reg, 'test MSE', 'Regression: optimization vs quality')
plot_dual_axis(h_clf, metric_clf, 'test Accuracy', 'Classification: optimization vs quality')